# "THE PRICE IS RIGHT" Capstone Project

This week - build a model that predicts how much something costs from a description, based on a scrape of Amazon data


A model that can estimate how much something costs, from its description.

# Order of play

DAY 1: Data Curation  
DAY 2: Data Pre-processing  
DAY 3: Evaluation, Baselines, Traditional ML  
DAY 4: Deep Learning and LLMs  
DAY 5: Fine-tuning a Frontier Model  

## DAY 2: Data Pre-processing

Today we'll rewrite the products into a standard format.  
LLMs are great at this!


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business value of Data Pre-processing / Re-writing</h2>
            <span style="color:#181;">LLMs have made it simple to do something that was considered impossible only a few years ago.
            This approach can be applied to almost any business vertical, and it's similar to the advanced techniques
            we used on Week 5.</span>
        </td>
    </tr>
</table>

In [1]:
from litellm import completion
from dotenv import load_dotenv
import json
# from pricer.batch import Batch
# from pricer.items import Item
from pricer_lcl.runner import Batch
from pricer.items import Item
import os

load_dotenv(override=True)

True

# The next cell is where you choose Dataset

Use `LITE_MODE = True` for the free, fast version with training data size of 20,000

USe `LITE_MODE =  False` for the powerful, full version with training data size of 800,000

## For this lab

You can skip altogether and load the dataset from HuggingFace: $0

You can run pre-processing for the lite dataset: under $1

You can run pre-processing for the full dataset: $30

In [2]:
LITE_MODE = True

In [3]:
username = "ed-donner"
dataset = f"{username}/items_raw_lite" if LITE_MODE else f"{username}/items_raw_full"

train, val, test = Item.from_hub(dataset)

items = train + val + test

print(f"Loaded {len(items):,} items")
print(items[0])

Loaded 22,000 items
title='Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only)' category='Tools_and_Home_Improvement' price=64.3 full='Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only)\n[\'From the Manufacturer\', "When you have a Schlage handleset on your front door, you ensure your security as well as your peace of mind. After all, we\'re the leader in security devices, trusted for over 85 years. All Schlage handlesets are precision engineered, featuring 100% solid"]\n[\'Interior half only\', \'Requires F58 to complete handle set\', \'Non handed knob style\', \'4" minimum center to center door prep required for this two piece model.\', \'Lifetime Mechanical and Finish Warranty\']\n{"Material": "Metal", "Brand": "", "Color": "Oil Rubbed Bronze", "Exterior Finish": "Bronze", "Special Feature": "Easy to Install", "Age Range (Description)": "Adult", "Included Components": "Deadbolt, Knob", "Item Wei

In [4]:
items[2].id

In [5]:
# Give every item an id

for index, item in enumerate(items):
    item.id = index

In [82]:


SYSTEM_PROMPT = """Create a concise description of a product. Respond only in this format. Do not include part numbers.
Title: Rewritten short precise title
Category: eg Electronics
Brand: Brand name
Description: 1 sentence description
Details: 1 sentence on features
"""

In [83]:
print(items[0].full)

Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only)
['From the Manufacturer', "When you have a Schlage handleset on your front door, you ensure your security as well as your peace of mind. After all, we're the leader in security devices, trusted for over 85 years. All Schlage handlesets are precision engineered, featuring 100% solid"]
['Interior half only', 'Requires F58 to complete handle set', 'Non handed knob style', '4" minimum center to center door prep required for this two piece model.', 'Lifetime Mechanical and Finish Warranty']
{"Material": "Metal", "Brand": "", "Color": "Oil Rubbed Bronze", "Exterior Finish": "Bronze", "Special Feature": "Easy to Install", "Age Range (Description)": "Adult", "Included Components": "Deadbolt, Knob", "Item Weight": "1.5 pounds", "Handle Material": "Bronze", "Package Type": "Standard Packaging", "Unit Count": "1.0 Count", "Number of Items": "1", "Manufacturer": "Schlage", "Product Dimensions": "8.1 x 4

In [127]:
messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": items[0].full}]
response = completion(messages=messages, model="openai/gpt-oss-20b", reasoning = {"effort": "minimal"}, api_key=os.environ.get("OPENROUTER_API_KEY"), base_url=os.environ.get("OPENROUTER_BASE_URL"))

print(response.choices[0].message.content)
print()
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")

# Get cost using the attribute that seems to exist: response.usage.cost
cost = getattr(response.usage, 'cost', None)
if cost is not None:
    print(f"Cost: {cost*100:.3f} cents")
else:
    print("Cost: not available")


Title: Schlage F59 Interior Knob with Deadbolt – Oil Rubbed Bronze (Half)  
Category: Hardware  
Brand: Schlage  
Description: A durable oil‑rubbed bronze interior knob with deadbolt, ideal for enhancing front‑door security.  
Details: Features a solid metal construction with a lifetime mechanical and finish warranty, easy installation, and requires the F58 for a complete handle set.

Input tokens: 447
Output tokens: 97
Cost: 0.004 cents


In [128]:
messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": items[0].full}]
response = completion(messages=messages, model="openai/openai/gpt-oss-20b", api_base="http://192.168.68.104:8000/v1")
print(response.choices[0].message.content)
print()
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
# Get cost using the attribute that seems to exist: response.usage.cost
cost = getattr(response.usage, 'cost', None)
if cost is not None:
    print(f"Cost: {cost*100:.3f} cents")
else:
    print("Cost: not available")


Title: Schlage F59 & 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only)  
Category: Door Hardware  
Brand: Schlage  
Description: An oil–rubbed bronze interior half knob that integrates a deadbolt for enhanced home security.  
Details: Features easy installation, solid metal construction, and a lifetime mechanical and finish warranty.

Input tokens: 446
Output tokens: 301
Cost: not available


In [129]:
MODEL = "openai/gpt-oss-20b"

In [130]:
SYSTEM_PROMPT

'Create a concise description of a product. Respond only in this format. Do not include part numbers.\nTitle: Rewritten short precise title\nCategory: eg Electronics\nBrand: Brand name\nDescription: 1 sentence description\nDetails: 1 sentence on features\n'

In [131]:
# def make_jsonl(item):
#     body = {"model": MODEL, "messages": [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": item.full}], "reasoning_effort": "low"}
#     line = {"custom_id": str(item.id), "method": "POST", "url": "/v1/chat/completions", "body": body}
#     return json.dumps(line)

def make_jsonl(item):
    body = {
        "model": MODEL,
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": item.full}
        ],
        "include_reasoning": False
    }
    line = {"custom_id": str(item.id), "method": "POST", "url": "/v1/chat/completions", "body": body}
    return json.dumps(line)

In [132]:
items[0]

<Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only) = $64.3>

In [133]:
make_jsonl(items[0])

'{"custom_id": "0", "method": "POST", "url": "/v1/chat/completions", "body": {"model": "openai/gpt-oss-20b", "messages": [{"role": "system", "content": "Create a concise description of a product. Respond only in this format. Do not include part numbers.\\nTitle: Rewritten short precise title\\nCategory: eg Electronics\\nBrand: Brand name\\nDescription: 1 sentence description\\nDetails: 1 sentence on features\\n"}, {"role": "user", "content": "Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only)\\n[\'From the Manufacturer\', \\"When you have a Schlage handleset on your front door, you ensure your security as well as your peace of mind. After all, we\'re the leader in security devices, trusted for over 85 years. All Schlage handlesets are precision engineered, featuring 100% solid\\"]\\n[\'Interior half only\', \'Requires F58 to complete handle set\', \'Non handed knob style\', \'4\\" minimum center to center door prep required for this two piec

In [134]:

def make_file(start, end, filename):
    batch_file = filename
    with open(batch_file, "w") as f:
        for i in range(start, end):
            f.write(make_jsonl(items[i]))
            f.write("\n")

In [136]:
make_file(0, 1000, "jsonl/0_1000_lcl.jsonl")

In [113]:
from openai import OpenAI

client = OpenAI(base_url="http://192.168.68.104:8000/v1")

In [1]:
# with open("jsonl/0_1000_lcl.jsonl", "rb") as f:
#     response = client.files.create(file=f, purpose="batch")
# response

In [2]:
# response[0]

In [3]:
# file_id = response.id
# file_id

In [4]:
# response = client.batches.create(completion_window="24h", endpoint="/v1/chat/completions", input_file_id=file_id)
# response

In [5]:
# result = client.batches.retrieve(response.id)
# result

In [6]:
# response = client.files.content(result.output_file_id)
# response.write_to_file("jsonl/batch_results.jsonl")

In [19]:
with open("./jsonl/0_1000_lcl_results.jsonl", "r") as f:
    for line in f:
        json_line = json.loads(line)
        id = int(json_line["custom_id"])
        summary = json_line["response"]["body"]["choices"][0]["message"]["content"]
        items[id].summary = summary


In [20]:
print(items[0].full)

Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only)
['From the Manufacturer', "When you have a Schlage handleset on your front door, you ensure your security as well as your peace of mind. After all, we're the leader in security devices, trusted for over 85 years. All Schlage handlesets are precision engineered, featuring 100% solid"]
['Interior half only', 'Requires F58 to complete handle set', 'Non handed knob style', '4" minimum center to center door prep required for this two piece model.', 'Lifetime Mechanical and Finish Warranty']
{"Material": "Metal", "Brand": "", "Color": "Oil Rubbed Bronze", "Exterior Finish": "Bronze", "Special Feature": "Easy to Install", "Age Range (Description)": "Adult", "Included Components": "Deadbolt, Knob", "Item Weight": "1.5 pounds", "Handle Material": "Bronze", "Package Type": "Standard Packaging", "Unit Count": "1.0 Count", "Number of Items": "1", "Manufacturer": "Schlage", "Product Dimensions": "8.1 x 4

In [35]:
print(items[1000].summary)

None


## I've put exactly this logic into a Batch class

- Divides items into groups of 1,000
- Kicks off batches for each
- Allows us to monitor and collect the results when complete

## COSTS

Using Groq, for me - this cost under $1 for the Lite dataset and under $30 for the big dataset

But you don't need to pay anything! In the next lab, you can load my pre-processed results

In [28]:
import importlib
import pricer_lcl.runner
importlib.reload(pricer_lcl.runner)
Batch = pricer_lcl.runner.Batch
Batch

pricer_lcl.runner.Batch

In [29]:
Batch.create(items, LITE_MODE)

Created 22 batches


In [30]:
Batch.run()

Processing batches:   0%|          | 0/22000 [00:00<?, ?req/s]

2026-01-18 02:29:08 - pricer_lcl.runner - ERROR - Request custom_id=7087 read error on attempt 1/3: 
2026-01-18 02:33:14 - pricer_lcl.runner - ERROR - Request custom_id=11131 read error on attempt 1/3: 
2026-01-18 02:36:17 - pricer_lcl.runner - ERROR - Request custom_id=14057 read error on attempt 1/3: 
2026-01-18 02:39:20 - pricer_lcl.runner - ERROR - Request custom_id=17115 read error on attempt 1/3: 
2026-01-18 02:41:22 - pricer_lcl.runner - ERROR - Request custom_id=19066 read error on attempt 1/3: 


Processed 22 batches


In [31]:
Batch.fetch()

  0%|          | 0/22 [00:00<?, ?it/s]

Finished 22 of 22 batches


In [32]:
for index, item in enumerate(items):
    if not item.summary:
        print(index)

13878


In [33]:
print(items[10234].summary)

Title: Red Barn Harvest Fabric Backdrop (84x60in)  
Category: Photography Backdrop  
Brand: Allenjoy  
Description: A high‑resolution 84×60‑inch fabric backdrop featuring a rustic red barn, pumpkin harvest, and haystack scene for wedding, holiday, and studio photography.  
Details: Crafted from durable, wrinkle‑resistant polyester with seamless edges, it’s reusable, easy to clean, and ready for instant photo‑ready setup.


In [34]:
# Remove the fields that we don't need in the hub

for item in items:
    item.full = None
    item.id = None

## Push the final dataset to the hub

If lite mode, we'll only push the lite dataset

If full mode, we'll push both datasets (in case you decide to use lite later)

In [35]:
username = "t3rmina1"
full = f"{username}/items_full"
lite = f"{username}/items_lite"

if LITE_MODE:
    train = items[:20_000]
    val = items[20_000:21_000]
    test = items[21_000:]
    Item.push_to_hub(lite, train, val, test)
else:
    train = items[:800_000]
    val = items[800_000:810_000]
    test = items[810_000:]
    Item.push_to_hub(full, train, val, test)

    train_lite = train[:20_000]
    val_lite = val[:1_000]
    test_lite = test[:1_000]
    Item.push_to_hub(lite, train_lite, val_lite, test_lite)

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/20 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

New Data Upload: |          |  0.00B /  0.00B            

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

## And here they are!

https://huggingface.co/datasets/ed-donner/items_lite

https://huggingface.co/datasets/ed-donner/items_full
